# LSTM for Shampoo Sales Prediction

In this exercise we use a **Long Short-Term Memory (LSTM)** network to forecast monthly shampoo sales from a univariate time series.

**Dataset:** `shampoo.csv` -- Monthly shampoo sales over 3 years (columns: `Month`, `Sales`).

**Learning objectives:**
1. Load and visualize a univariate time series.
2. Normalize data with `MinMaxScaler`, fitted on the training months only.
3. Create sliding-window sequences for supervised learning.
4. Build and train an LSTM model in PyTorch.
5. Evaluate predictions on the test months and compare the RMSE with a naive (previous-month) baseline.

**Evaluation design.** The last 6 months are the test period. Every fitted step (here the scaler and the network) uses the earlier months only, and the test months are scored once, at the end. With 36 values there is no room for a validation period, so the settings (window length, network size, number of epochs) are fixed in advance, not tuned on the test months. Runtime: a few seconds on a laptop CPU.

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)  # initial weights, dropout and DataLoader shuffling

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## 2. Load and Explore the Data

In [ ]:
df = pd.read_csv("shampoo.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
df.describe()

---
## 3. Visualize the Time Series

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["Sales"].values, marker="o", markersize=3)
plt.title("Monthly Shampoo Sales")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Normalize Data with MinMaxScaler

Fit the scaler on the **training months only** and apply it unchanged to every month. With a 6-month window there are 30 windows; the first 80% (24) are for training, and their inputs and targets are the first 6 + 24 = 30 months. Fitting on all 36 months would leak the range of the test months into training. Test values may then fall outside [0, 1]; that is expected, so do not refit or clip.

In [ ]:
sales = df["Sales"].values.reshape(-1, 1).astype(np.float32)

scaler = MinMaxScaler(feature_range=(0, 1))
# Match the later 80/20 chronological split of one-step windows.
SEQ_LENGTH = 6
train_sequence_count = int((len(sales) - SEQ_LENGTH) * 0.8)
training_raw_end = SEQ_LENGTH + train_sequence_count
scaler.fit(sales[:training_raw_end])  # training months only
sales_scaled = scaler.transform(sales)
# Later observed values may fall outside [0, 1]; do not refit or clip them.

print(f"Scaler fitted on months 1-{training_raw_end} of {len(sales)}")
print(f"Original range: [{sales.min():.1f}, {sales.max():.1f}]")
print(f"Scaled range:   [{sales_scaled.min():.4f}, {sales_scaled.max():.4f}]")

---
## 5. Create Sequences (Sliding Window)

In [ ]:
def create_sequences(data, seq_length):
    """Create input/target pairs using a sliding window.
    
    Args:
        data: 1D array of scaled values
        seq_length: number of past time steps to use as input
    
    Returns:
        X: array of shape (num_samples, seq_length, 1)
        y: array of shape (num_samples, 1)
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)


SEQ_LENGTH = 6  # use past 6 months to predict the next month

X, y = create_sequences(sales_scaled, SEQ_LENGTH)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

---
## 6. Train/Test Split and PyTorch DataLoaders

Each window belongs to the period that contains its **target** month. The 24 training windows have targets in months 7-30; the 6 test windows have targets in months 31-36. A test window may read months before month 31 as inputs: those values are already observed at the forecast origin, which is what a deployed forecast would see. No training target lies in the test period, and the scaler saw exactly the training months.

In [ ]:
# Time series split -- use the last 20% of windows (the last 6 target months) as test
split = int(len(X) * 0.8)
assert split + SEQ_LENGTH == training_raw_end  # scaler fitted on the training months only

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

BATCH_SIZE = 8
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"Test target months: {split + SEQ_LENGTH + 1}-{len(sales)}")
print(f"Input tensor shape per batch: (batch, {SEQ_LENGTH}, 1)")

---
## 7. Build the LSTM Model

In [ ]:
class ShampooLSTM(nn.Module):
    """Simple LSTM for univariate time series forecasting."""

    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, seq_length, input_size)
        lstm_out, _ = self.lstm(x)  # (batch, seq_length, hidden_size)
        last_hidden = lstm_out[:, -1, :]  # take the last time step
        out = self.fc(last_hidden)  # (batch, 1)
        return out


model = ShampooLSTM().to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 8. Train the Model

`NUM_EPOCHS = 150` is fixed in advance; it is not chosen by looking at the test months. With a validation period you would pick the epoch there, as the climate notebook does with early stopping.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 150
train_losses = []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)

    epoch_loss = running_loss / len(train_ds)
    train_losses.append(epoch_loss)

    if epoch % 25 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS}  Loss: {epoch_loss:.6f}")

---
## 9. Plot the Training Loss Curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.title("Training Loss (MSE)")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Predict on Test Data and Inverse Transform

This is the final evaluation: the six test months are used once, for the model trained above.

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t.to(DEVICE)).cpu().numpy()

# Inverse transform to original scale
y_pred = scaler.inverse_transform(y_pred_scaled)
y_actual = scaler.inverse_transform(y_test)

print(f"Predicted shape: {y_pred.shape}")
print(f"Actual shape:    {y_actual.shape}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(y_actual, label="Actual", marker="o", markersize=4)
plt.plot(y_pred, label="Predicted", marker="s", markersize=4)
plt.title("Shampoo Sales: Predicted vs Actual")
plt.xlabel("Test Sample")
plt.ylabel("Sales")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 11. Compute RMSE

The naive baseline repeats the last observed month and is scored on exactly the same six target months as the LSTM. Six targets are far too few to rank models: treat the comparison as a mechanics check, not as evidence that one method is better.

In [ ]:
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

# Naive baseline on the same test months: predict the previous month's value
naive_pred = scaler.inverse_transform(X_test[:, -1, :])  # last value in each window
naive_rmse = np.sqrt(mean_squared_error(y_actual, naive_pred))

print(f"Naive baseline RMSE (previous month): {naive_rmse:.1f}")
print(f"LSTM test RMSE:                       {rmse:.1f}")
print(f"Improvement over naive: {(naive_rmse - rmse) / naive_rmse * 100:.1f}%")

---
## 12. Summary

In this notebook we:

1. Loaded the shampoo sales dataset -- a simple univariate monthly time series.
2. Normalized the data with a `MinMaxScaler` fitted on the training months only.
3. Created sliding-window sequences (past 6 months to predict the next month).
4. Split the windows by target month: 24 for training, the last 6 for testing.
5. Built a 2-layer LSTM with a fully connected output head and trained it for a fixed number of epochs.
6. Predicted the test months once, inverse-transformed, and plotted results.
7. Computed RMSE and compared it with a naive baseline on the same months.

**Key takeaways:**
- LSTMs are well-suited for sequential/temporal data because they maintain memory of past inputs.
- Proper normalization is essential for stable LSTM training -- and the scaler must be fitted on training data only.
- The sliding-window approach converts a time series into a supervised learning problem.
- Always compare your model to a simple baseline (e.g., naive persistence) on the same targets; with six test months, treat the result as a mechanics check.